# Public Transport Delay Prediction using Machine Learning

## MLflow Experiment Tracking

### Objective

Track the completed experiments from Notebooks 3, 4, and 5 using **MLflow**.

### Important MLOps design

This notebook does **not** manually type model metrics.

The modelling notebooks are the source of truth:

```text
Notebook 3 → classification result CSVs
Notebook 4 → regression result CSVs
Notebook 5 → time-series result CSVs
Notebook 6 → explainability artefacts
                         ↓
                    Notebook 7
                         ↓
                       MLflow
```

If a model is rerun and its metrics change, Notebook 7 reads the updated result files.

### Tasks tracked

- Classification
- Regression
- Time-series forecasting
- Regression explainability artefacts

> This notebook tracks completed experiments. It does not retrain the models.


In [1]:

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

print("MLflow imported successfully.")
print("MLflow version:", mlflow.__version__)


MLflow imported successfully.
MLflow version: 3.16.1


## Configure MLflow Tracking

A local SQLite database stores the MLflow tracking metadata.

Using SQLite keeps the project self-contained during development. The tracking backend can later be moved to a shared MLflow server for team or production use.

In [2]:
PROJECT_ROOT = Path("..").resolve()

MLFLOW_DB_PATH = PROJECT_ROOT / "mlflow.db"

TRACKING_URI = f"sqlite:///{MLFLOW_DB_PATH.as_posix()}"

mlflow.set_tracking_uri(TRACKING_URI)

EXPERIMENT_NAME = "Public Transport Delay Prediction - MLOps"

mlflow.set_experiment(EXPERIMENT_NAME)

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name)
print("Experiment ID:", experiment.experiment_id)


Tracking URI: sqlite:///C:/Users/srisa/OneDrive/Desktop/Projects/Public Transport Delays/mlflow.db
Experiment: Public Transport Delay Prediction - MLOps
Experiment ID: 2


In [3]:
# Project directories

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = ARTIFACTS_DIR / "models"
SPLITS_DIR = ARTIFACTS_DIR / "splits"

RESULTS_DIR = PROJECT_ROOT / "results"
CLASSIFICATION_RESULTS_DIR = RESULTS_DIR / "classification"
REGRESSION_RESULTS_DIR = RESULTS_DIR / "regression"
TIME_SERIES_RESULTS_DIR = RESULTS_DIR / "time_series"
EXPLAINABILITY_RESULTS_DIR = RESULTS_DIR / "explainability"
MLFLOW_RESULTS_DIR = RESULTS_DIR / "mlflow"

MLFLOW_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

REGRESSION_MODEL_PATH = (
    MODELS_DIR / "best_regression_pipeline.joblib"
)

TIME_SERIES_MODEL_PATH = (
    MODELS_DIR / "best_time_series_model.joblib"
)

print("Project root:", PROJECT_ROOT)
print("Regression model exists:", REGRESSION_MODEL_PATH.exists())
print("Time-series model exists:", TIME_SERIES_MODEL_PATH.exists())
print("Classification results exist:", CLASSIFICATION_RESULTS_DIR.exists())
print("Regression results exist:", REGRESSION_RESULTS_DIR.exists())
print("Time-series results exist:", TIME_SERIES_RESULTS_DIR.exists())
print("Explainability results exist:", EXPLAINABILITY_RESULTS_DIR.exists())


Project root: C:\Users\srisa\OneDrive\Desktop\Projects\Public Transport Delays
Regression model exists: True
Time-series model exists: True
Classification results exist: True
Regression results exist: True
Time-series results exist: True
Explainability results exist: True


## Helper Functions

These functions only handle MLflow logging.

They do not manufacture or modify modelling results.

In [4]:
def start_logged_run(run_name, task, source_notebook, tags=None):
    base_tags = {
        "project": "public_transport_delay_prediction",
        "task": task,
        "source_notebook": source_notebook,
        "tracking_type": "completed_experiment",
    }

    if tags:
        base_tags.update(tags)

    return mlflow.start_run(
        run_name=run_name,
        tags=base_tags,
    )


def log_metric_dict(metrics):
    for name, value in metrics.items():
        if value is not None and pd.notna(value):
            mlflow.log_metric(name, float(value))


def log_existing_file(path, artifact_path=None):
    path = Path(path)

    if not path.exists():
        print("Skipped missing artefact:", path)
        return False

    mlflow.log_artifact(
        str(path),
        artifact_path=artifact_path,
    )
    return True


print("MLflow helper functions created.")


MLflow helper functions created.


# 1. Classification Experiment Tracking

Notebook 3 identified an important dataset limitation: the binary `delayed` target exactly matches whether `actual_arrival_delay_min > 5`.

Therefore, classification is tracked as a **diagnostic experiment**, not as a production-ready pre-trip classifier.


In [5]:
CLASSIFICATION_CV_PATH = (
    CLASSIFICATION_RESULTS_DIR
    / "classification_cv_results.csv"
)

CLASSIFICATION_TEST_PATH = (
    CLASSIFICATION_RESULTS_DIR
    / "classification_test_metrics.csv"
)

required_paths = [
    CLASSIFICATION_CV_PATH,
    CLASSIFICATION_TEST_PATH,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing classification result file: {path}"
        )

classification_cv = pd.read_csv(
    CLASSIFICATION_CV_PATH
)

classification_test = pd.read_csv(
    CLASSIFICATION_TEST_PATH
)

required_cv_columns = {
    "Model",
    "CV Accuracy",
    "CV Balanced Accuracy",
    "CV Precision",
    "CV Recall",
    "CV F1",
    "CV ROC-AUC",
    "CV PR-AUC",
}

required_test_columns = {
    "Model",
    "Accuracy",
    "Balanced Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "PR-AUC",
}

if not required_cv_columns.issubset(classification_cv.columns):
    raise ValueError(
        "Classification CV CSV has unexpected/missing columns."
    )

if not required_test_columns.issubset(classification_test.columns):
    raise ValueError(
        "Classification test CSV has unexpected/missing columns."
    )

display(classification_cv.round(4))
display(classification_test.round(4))


,Model,CV Accuracy,CV Balanced Accuracy,CV Precision,CV Recall,CV F1,CV ROC-AUC,CV PR-AUC
0,Random Forest,0.7494,0.5000,0.7494,1.0000,0.8567,0.5038,0.7540
1,Extra Trees,0.7488,0.5071,0.7521,0.9917,0.8554,0.4974,0.7532
2,KNN,0.7369,0.4933,0.7468,0.9816,0.8483,0.4875,0.7528
3,Gradient Boosting,0.7294,0.4975,0.7484,0.9625,0.8420,0.5005,0.7488
4,XGBoost,0.6150,0.4883,0.7436,0.7423,0.7428,0.4946,0.7441
5,SVM,0.5694,0.4986,0.7479,0.6405,0.6895,0.5075,0.7676
6,Logistic Regression,0.5312,0.4914,0.7439,0.5713,0.6462,0.4703,0.7301
7,Gaussian Naive Bayes,0.5069,0.4925,0.7437,0.5213,0.6118,0.4956,0.7505
8,Decision Tree,0.4625,0.4904,0.7422,0.4346,0.5425,0.4953,0.7473


,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Random Forest,0.75,0.5,0.75,1.0,0.8571,0.4818,0.7298


In [6]:
# Read classification dataset dimensions from the saved splits.

X_train_cls = pd.read_csv(
    SPLITS_DIR / "X_train_cls.csv"
)

X_test_cls = pd.read_csv(
    SPLITS_DIR / "X_test_cls.csv"
)

print("Classification training rows:", len(X_train_cls))
print("Classification test rows:", len(X_test_cls))
print("Classification input features:", X_train_cls.shape[1])


Classification training rows: 1600
Classification test rows: 400
Classification input features: 33


In [7]:
# Log classification CV runs directly from the CSV created by Notebook 3.

for _, row in classification_cv.iterrows():

    with start_logged_run(
        f"Classification - {row['Model']} CV",
        task="classification",
        source_notebook="03_Classification_Model_Building_and_Evaluation",
        tags={
            "stage": "model_comparison",
            "evaluation_period": "5_fold_cross_validation",
            "status": "diagnostic",
            "target": "delayed",
        },
    ):

        mlflow.log_params({
            "model": row["Model"],
            "cv_folds": 5,
            "input_features": X_train_cls.shape[1],
            "training_rows": len(X_train_cls),
            "test_rows": len(X_test_cls),
        })

        log_metric_dict({
            "cv_accuracy": row["CV Accuracy"],
            "cv_balanced_accuracy": row["CV Balanced Accuracy"],
            "cv_precision": row["CV Precision"],
            "cv_recall": row["CV Recall"],
            "cv_f1": row["CV F1"],
            "cv_roc_auc": row["CV ROC-AUC"],
            "cv_pr_auc": row["CV PR-AUC"],
        })

print("Classification CV experiments logged.")


Classification CV experiments logged.


In [8]:
# Log the final Random Forest classification diagnostic result.

classification_test_row = classification_test.iloc[0]

with start_logged_run(
    "Classification - Random Forest Diagnostic Final",
    task="classification",
    source_notebook="03_Classification_Model_Building_and_Evaluation",
    tags={
        "stage": "final_test_evaluation",
        "status": "diagnostic",
        "target": "delayed",
        "reason": "target_construction_limitation",
    },
):

    mlflow.log_params({
        "model": classification_test_row["Model"],
        "input_features": X_train_cls.shape[1],
        "training_rows": len(X_train_cls),
        "test_rows": len(X_test_cls),
        "cv_folds": 5,
        "target_rule": (
            "delayed == (actual_arrival_delay_min > 5)"
        ),
    })

    log_metric_dict({
        "test_accuracy": classification_test_row["Accuracy"],
        "test_balanced_accuracy": classification_test_row["Balanced Accuracy"],
        "test_precision": classification_test_row["Precision"],
        "test_recall": classification_test_row["Recall"],
        "test_f1": classification_test_row["F1"],
        "test_roc_auc": classification_test_row["ROC-AUC"],
        "test_pr_auc": classification_test_row["PR-AUC"],
    })

    log_existing_file(
        CLASSIFICATION_CV_PATH,
        "evaluation",
    )

    log_existing_file(
        CLASSIFICATION_TEST_PATH,
        "evaluation",
    )

    mlflow.log_text(
        "Notebook 3 found that delayed exactly matches "
        "whether actual_arrival_delay_min exceeds 5 minutes. "
        "Post-outcome delay variables were excluded from the "
        "strict pre-trip classification scenario.",
        "classification_finding.txt",
    )

print("Classification diagnostic run logged.")

Classification diagnostic run logged.


# 2. Regression Experiment Tracking

Notebook 4 is the main continuous-delay prediction workflow.

All model results and tuning parameters are read from the CSV files generated by Notebook 4.


In [9]:
REGRESSION_CV_PATH = (
    REGRESSION_RESULTS_DIR
    / "regression_cv_results.csv"
)

REGRESSION_TUNING_PATH = (
    REGRESSION_RESULTS_DIR
    / "regression_tuning_results.csv"
)

REGRESSION_TEST_PATH = (
    REGRESSION_RESULTS_DIR
    / "regression_test_metrics.csv"
)

REGRESSION_PREDICTIONS_PATH = (
    REGRESSION_RESULTS_DIR
    / "regression_test_predictions.csv"
)

for path in [
    REGRESSION_CV_PATH,
    REGRESSION_TUNING_PATH,
    REGRESSION_TEST_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing regression result file: {path}"
        )

regression_cv = pd.read_csv(
    REGRESSION_CV_PATH
)

regression_tuning = pd.read_csv(
    REGRESSION_TUNING_PATH
)

regression_test = pd.read_csv(
    REGRESSION_TEST_PATH
)

required_cv_columns = {
    "Model",
    "CV MAE",
    "CV RMSE",
    "CV R2",
}

required_tuning_columns = {
    "Best CV RMSE",
    "model__max_depth",
    "model__max_features",
    "model__min_samples_leaf",
    "model__n_estimators",
}

required_test_columns = {
    "Model",
    "MAE",
    "RMSE",
    "R2",
}

if not required_cv_columns.issubset(regression_cv.columns):
    raise ValueError(
        "Regression CV CSV has unexpected/missing columns."
    )

if not required_tuning_columns.issubset(
    regression_tuning.columns
):
    raise ValueError(
        "Regression tuning CSV has unexpected/missing columns."
    )

if not required_test_columns.issubset(regression_test.columns):
    raise ValueError(
        "Regression test CSV has unexpected/missing columns."
    )

display(regression_cv.round(4))
display(regression_tuning.round(4))
display(regression_test.round(4))


,Model,CV MAE,CV RMSE,CV R2
0,Mean Baseline,8.0218,9.3145,-0.0048
1,Random Forest,8.0042,9.3649,-0.0159
2,Extra Trees,8.1366,9.4999,-0.0452
3,Gradient Boosting,8.1890,9.5653,-0.0596
4,XGBoost,8.3121,9.7667,-0.1049
5,Lasso,8.3089,9.7876,-0.1094
6,Ridge,8.4231,9.9693,-0.1513
7,Linear Regression,8.4538,10.0169,-0.1623


,Best CV RMSE,model__max_depth,model__max_features,model__min_samples_leaf,model__n_estimators
0,9.3056,10,log2,5,200


,Model,MAE,RMSE,R2
0,Mean Baseline,7.7030,9.1901,-0.0017
1,Tuned Random Forest,7.7501,9.2602,-0.0170


In [10]:
# Read regression train/test dimensions from Notebook 2 splits.

X_train_reg = pd.read_csv(
    SPLITS_DIR / "X_train_reg.csv"
)

X_test_reg = pd.read_csv(
    SPLITS_DIR / "X_test_reg.csv"
)

y_test_reg = pd.read_csv(
    SPLITS_DIR / "y_test_reg.csv"
).squeeze("columns")

print("Regression training rows:", len(X_train_reg))
print("Regression test rows:", len(X_test_reg))
print("Regression input features:", X_train_reg.shape[1])
print("Regression target rows:", len(y_test_reg))


Regression training rows: 1600
Regression test rows: 400
Regression input features: 33
Regression target rows: 400


In [11]:
# Log every regression CV result from Notebook 4.

for _, row in regression_cv.iterrows():

    with start_logged_run(
        f"Regression - {row['Model']} CV",
        task="regression",
        source_notebook="04_Regression_Model_Building_and_Evaluation",
        tags={
            "stage": "model_comparison",
            "evaluation_period": "5_fold_cross_validation",
            "target": "actual_arrival_delay_min",
        },
    ):

        mlflow.log_params({
            "model": row["Model"],
            "cv_folds": 5,
            "selection_metric": "RMSE",
            "input_features": X_train_reg.shape[1],
            "training_rows": len(X_train_reg),
            "test_rows": len(X_test_reg),
        })

        log_metric_dict({
            "cv_mae": row["CV MAE"],
            "cv_rmse": row["CV RMSE"],
            "cv_r2": row["CV R2"],
        })

print("Regression CV experiments logged.")


Regression CV experiments logged.


## Verify the Final Regression Result

Before logging the final regression experiment, the test metrics stored by Notebook 4 are checked against predictions generated by the saved pipeline.

This prevents a stale or manually edited CSV from being logged as the model's result.


In [12]:
if not REGRESSION_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Saved regression model not found: {REGRESSION_MODEL_PATH}"
    )

final_regression_pipeline = joblib.load(
    REGRESSION_MODEL_PATH
)

saved_predictions = final_regression_pipeline.predict(
    X_test_reg
)

recomputed_metrics = {
    "MAE": mean_absolute_error(
        y_test_reg,
        saved_predictions
    ),
    "RMSE": np.sqrt(
        mean_squared_error(
            y_test_reg,
            saved_predictions
        )
    ),
    "R2": r2_score(
        y_test_reg,
        saved_predictions
    ),
}

regression_test_row = regression_test.loc[
    regression_test["Model"].astype(str).str.strip()
    == "Tuned Random Forest"
]

if len(regression_test_row) != 1:
    raise ValueError(
        "Expected exactly one Tuned Random Forest row "
        "in regression_test_metrics.csv."
    )

regression_test_row = regression_test_row.iloc[0]

metric_checks = {
    metric: np.isclose(
        recomputed_metrics[metric],
        float(regression_test_row[metric]),
        rtol=1e-6,
        atol=1e-6,
    )
    for metric in ["MAE", "RMSE", "R2"]
}

print(
    "Recomputed model metrics:",
    {k: round(v, 4) for k, v in recomputed_metrics.items()}
)

print(
    "Saved CSV metrics:",
    {
        k: round(float(regression_test_row[k]), 4)
        for k in ["MAE", "RMSE", "R2"]
    }
)

print("Metric checks:", metric_checks)

if not all(metric_checks.values()):
    raise ValueError(
        "Regression CSV metrics do not match "
        "predictions from the saved model."
    )

print("Regression metric consistency check passed.")


Recomputed model metrics: {'MAE': 7.7501, 'RMSE': 9.2602, 'R2': -0.017}
Saved CSV metrics: {'MAE': 7.7501, 'RMSE': 9.2602, 'R2': -0.017}
Metric checks: {'MAE': True, 'RMSE': True, 'R2': True}
Regression metric consistency check passed.


In [13]:
# Read tuning parameters directly from Notebook 4.

tuning_row = regression_tuning.iloc[0]

tuned_parameters = {
    "max_depth": int(
        tuning_row["model__max_depth"]
    ),
    "max_features": str(
        tuning_row["model__max_features"]
    ),
    "min_samples_leaf": int(
        tuning_row["model__min_samples_leaf"]
    ),
    "n_estimators": int(
        tuning_row["model__n_estimators"]
    ),
}

rf_model = final_regression_pipeline.named_steps["model"]

parameter_checks = {
    "max_depth": (
        rf_model.get_params()["max_depth"]
        == tuned_parameters["max_depth"]
    ),
    "max_features": (
        rf_model.get_params()["max_features"]
        == tuned_parameters["max_features"]
    ),
    "min_samples_leaf": (
        rf_model.get_params()["min_samples_leaf"]
        == tuned_parameters["min_samples_leaf"]
    ),
    "n_estimators": (
        rf_model.get_params()["n_estimators"]
        == tuned_parameters["n_estimators"]
    ),
}

print("Tuning file parameters:", tuned_parameters)
print("Saved model parameters:", {
    key: rf_model.get_params()[key]
    for key in tuned_parameters
})
print("Parameter checks:", parameter_checks)

if not all(parameter_checks.values()):
    raise ValueError(
        "Regression tuning parameters do not match "
        "the saved Random Forest."
    )

print("Regression parameter consistency check passed.")


Tuning file parameters: {'max_depth': 10, 'max_features': 'log2', 'min_samples_leaf': 5, 'n_estimators': 200}
Saved model parameters: {'max_depth': 10, 'max_features': 'log2', 'min_samples_leaf': 5, 'n_estimators': 200}
Parameter checks: {'max_depth': True, 'max_features': True, 'min_samples_leaf': True, 'n_estimators': True}
Regression parameter consistency check passed.


In [14]:
# Read CV comparison values directly from the CV CSV.

baseline_cv_rmse = float(
    regression_cv.loc[
        regression_cv["Model"].astype(str).str.strip()
        == "Mean Baseline",
        "CV RMSE",
    ].iloc[0]
)

untuned_rf_cv_rmse = float(
    regression_cv.loc[
        regression_cv["Model"].astype(str).str.strip()
        == "Random Forest",
        "CV RMSE",
    ].iloc[0]
)

tuned_rf_cv_rmse = float(
    tuning_row["Best CV RMSE"]
)

print(
    "Baseline CV RMSE:",
    round(baseline_cv_rmse, 4)
)

print(
    "Untuned Random Forest CV RMSE:",
    round(untuned_rf_cv_rmse, 4)
)

print(
    "Tuned Random Forest CV RMSE:",
    round(tuned_rf_cv_rmse, 4)
)

Baseline CV RMSE: 9.3145
Untuned Random Forest CV RMSE: 9.3649
Tuned Random Forest CV RMSE: 9.3056


## Log the Final Regression Run

The final run records the actual tuning parameters, CV metrics, test metrics, saved model, evaluation CSVs, and explainability artefacts.

The model itself is also logged using MLflow's scikit-learn integration with pickle serialization. This avoids the `skops` trusted-type issue encountered in the local environment. Pickle-based model artefacts should only be loaded from trusted sources.

In [15]:
signature = infer_signature(
    X_test_reg,
    saved_predictions
)

with start_logged_run(
    "Regression - Tuned Random Forest Final",
    task="regression",
    source_notebook="04_Regression_Model_Building_and_Evaluation",
    tags={
        "stage": "final_test_evaluation",
        "status": "final",
        "target": "actual_arrival_delay_min",
        "leakage_policy": "post_outcome_delay_excluded",
        "explainability": (
            "SHAP,permutation,feature_importance"
        ),
    },
) as final_regression_run:

    mlflow.log_params({
        **tuned_parameters,
        "input_features": X_train_reg.shape[1],
        "training_rows": len(X_train_reg),
        "test_rows": len(X_test_reg),
        "cv_folds": 5,
        "selection_metric": "RMSE",
    })

    log_metric_dict({
        "baseline_cv_rmse": baseline_cv_rmse,
        "untuned_rf_cv_rmse": untuned_rf_cv_rmse,
        "tuned_rf_cv_rmse": tuned_rf_cv_rmse,
        "test_mae": float(regression_test_row["MAE"]),
        "test_rmse": float(regression_test_row["RMSE"]),
        "test_r2": float(regression_test_row["R2"]),
    })

    # Original model file
    log_existing_file(
        REGRESSION_MODEL_PATH,
        "model_joblib",
    )

    # MLflow model
    mlflow.sklearn.log_model(
        final_regression_pipeline,
        name="regression_model",
        serialization_format="pickle",
        signature=signature,
        input_example=X_test_reg.head(3),
    )

    # Evaluation artefacts
    log_existing_file(
        REGRESSION_CV_PATH,
        "evaluation",
    )

    log_existing_file(
        REGRESSION_TUNING_PATH,
        "evaluation",
    )

    log_existing_file(
        REGRESSION_TEST_PATH,
        "evaluation",
    )

    log_existing_file(
        REGRESSION_PREDICTIONS_PATH,
        "evaluation",
    )

    # Explainability artefacts from Notebook 6
    for file_path in sorted(
        EXPLAINABILITY_RESULTS_DIR.glob("*")
    ):
        if file_path.is_file():
            log_existing_file(
                file_path,
                "explainability",
            )
    
    final_regression_run_id = final_regression_run.info.run_id
print("Final regression experiment logged.")


2026/10/05 21:45:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Final regression experiment logged.


In [16]:
# Register the final regression model

REGISTERED_MODEL_NAME = "public-transport-delay-regressor"

model_uri = (
    f"runs:/{final_regression_run_id}/regression_model"
)

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME,
)

print("Registered model:", registered_model.name)
print("Model version:", registered_model.version)
print("Source run:", registered_model.run_id)

Successfully registered model 'public-transport-delay-regressor'.
2026/10/05 21:45:34 WARNING mlflow.tracking._model_registry.fluent: Run with id 0db24055614f44419edf1bcffdc88577 has no artifacts at artifact path 'regression_model', registering model based on models:/m-c1041b0dd18c417899f09281451abad8 instead


Registered model: public-transport-delay-regressor
Model version: 1
Source run: 0db24055614f44419edf1bcffdc88577


Created version '1' of model 'public-transport-delay-regressor'.


In [17]:
# Assign the registered model version to the production alias

client = MlflowClient()

client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="production",
    version=registered_model.version,
)

print(
    f"Model version {registered_model.version} "
    f"is now assigned to alias '@production'."
)

Model version 1 is now assigned to alias '@production'.


In [ ]:
# Verify the production model can be loaded

production_model_uri = (
    f"models:/{REGISTERED_MODEL_NAME}@production"
)

production_model = mlflow.pyfunc.load_model(
    production_model_uri
)

production_predictions = production_model.predict(
    X_test_reg.head(5)
)

print("Production model URI:")
print(production_model_uri)

print("\nSample predictions:")
print(production_predictions)

#check the registry/production alias
print("\n Registered model:",client.get_model_version_by_alias(
    REGISTERED_MODEL_NAME,
    "production"
))


Production model URI:
models:/public-transport-delay-regressor@production

Sample predictions:
[14.5532437  13.68384852 14.22526653 14.20687173 13.86593709]

 Registered model: <ModelVersion: aliases=['production'], creation_timestamp=1791233134103, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1791233134103, metrics=None, model_id=None, name='public-transport-delay-regressor', params=None, run_id='0db24055614f44419edf1bcffdc88577', run_link=None, source='models:/m-c1041b0dd18c417899f09281451abad8', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

 Model version by Alias <ModelVersion: aliases=['production'], creation_timestamp=1791233134103, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1791233134103, metrics=None, model_id=None, name='public-transport-delay-regressor', params=None, run_id='0db24055614f44419edf1bcffdc88577', run_link=None, source='models:

# 3. Time-Series Experiment Tracking

Notebook 5 compared short-term forecasting approaches using chronological validation.

The validation and test results are read directly from Notebook 5's CSV files.

In [19]:
TIME_SERIES_VALIDATION_PATH = (
    TIME_SERIES_RESULTS_DIR
    / "time_series_validation_results.csv"
)

TIME_SERIES_TEST_PATH = (
    TIME_SERIES_RESULTS_DIR
    / "time_series_test_results.csv"
)

TIME_SERIES_FORECAST_PATH = (
    TIME_SERIES_RESULTS_DIR
    / "time_series_forecast_results.csv"
)

for path in [
    TIME_SERIES_VALIDATION_PATH,
    TIME_SERIES_TEST_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing time-series result file: {path}"
        )

time_series_validation = pd.read_csv(
    TIME_SERIES_VALIDATION_PATH
)

time_series_test = pd.read_csv(
    TIME_SERIES_TEST_PATH
)

required_columns = {"Model", "MAE", "RMSE"}

if not required_columns.issubset(
    time_series_validation.columns
):
    raise ValueError(
        "Time-series validation CSV has "
        "unexpected/missing columns."
    )

if not required_columns.issubset(
    time_series_test.columns
):
    raise ValueError(
        "Time-series test CSV has "
        "unexpected/missing columns."
    )

display(time_series_validation.round(4))
display(time_series_test.round(4))

,Model,MAE,RMSE
0,3-Day Moving Average,0.2811,0.4014
1,Simple Exponential Smoothing,0.4802,0.6824
2,"ARIMA(1,0,0)",0.4864,0.6941
3,Naive Last Value,0.5833,0.7822


,Model,MAE,RMSE
0,3-Day Moving Average,0.9604,1.0946


In [20]:
# Select the model from the actual validation results.

selected_ts_row = (
    time_series_validation
    .sort_values("RMSE")
    .iloc[0]
)

selected_ts_model = str(
    selected_ts_row["Model"]
).strip()

final_ts_row = time_series_test.loc[
    time_series_test["Model"].astype(str).str.strip()
    == selected_ts_model
]

if len(final_ts_row) != 1:
    raise ValueError(
        "The selected time-series model does not have "
        "exactly one final test result."
    )

final_ts_row = final_ts_row.iloc[0]

print("Selected model:", selected_ts_model)
print(
    "Validation MAE:",
    round(float(selected_ts_row["MAE"]), 4)
)
print(
    "Validation RMSE:",
    round(float(selected_ts_row["RMSE"]), 4)
)
print(
    "Test MAE:",
    round(float(final_ts_row["MAE"]), 4)
)
print(
    "Test RMSE:",
    round(float(final_ts_row["RMSE"]), 4)
)


Selected model: 3-Day Moving Average
Validation MAE: 0.2811
Validation RMSE: 0.4014
Test MAE: 0.9604
Test RMSE: 1.0946


In [21]:
# Log every time-series validation experiment.

for _, row in time_series_validation.iterrows():

    model_name = str(row["Model"]).strip()

    params = {
        "model": model_name,
        "frequency": "D",
        "selection_metric": "RMSE",
    }

    if model_name == "3-Day Moving Average":
        params["window"] = 3

    if model_name == "ARIMA(1,0,0)":
        params["order"] = "(1,0,0)"

    with start_logged_run(
        f"Time Series - {model_name} Validation",
        task="time_series",
        source_notebook="05_Time_Series_Analysis_and_Forecasting",
        tags={
            "stage": "model_comparison",
            "evaluation_period": "chronological_validation",
            "target": "daily_mean_arrival_delay_min",
        },
    ):

        mlflow.log_params(params)

        log_metric_dict({
            "validation_mae": row["MAE"],
            "validation_rmse": row["RMSE"],
        })

        log_existing_file(
            TIME_SERIES_VALIDATION_PATH,
            "evaluation",
        )

print("Time-series validation experiments logged.")


Time-series validation experiments logged.


In [22]:
# Log the final selected time-series model.

final_ts_params = {
    "model": selected_ts_model,
    "frequency": "D",
    "selection_metric": "RMSE",
}

if selected_ts_model == "3-Day Moving Average":
    final_ts_params["window"] = 3

if selected_ts_model == "ARIMA(1,0,0)":
    final_ts_params["order"] = "(1,0,0)"

with start_logged_run(
    f"Time Series - {selected_ts_model} Final",
    task="time_series",
    source_notebook="05_Time_Series_Analysis_and_Forecasting",
    tags={
        "stage": "final_test_evaluation",
        "status": "final",
        "target": "daily_mean_arrival_delay_min",
    },
):

    mlflow.log_params(final_ts_params)

    log_metric_dict({
        "validation_mae": float(
            selected_ts_row["MAE"]
        ),
        "validation_rmse": float(
            selected_ts_row["RMSE"]
        ),
        "test_mae": float(
            final_ts_row["MAE"]
        ),
        "test_rmse": float(
            final_ts_row["RMSE"]
        ),
    })

    log_existing_file(
        TIME_SERIES_MODEL_PATH,
        "model",
    )

    log_existing_file(
        TIME_SERIES_VALIDATION_PATH,
        "evaluation",
    )

    log_existing_file(
        TIME_SERIES_TEST_PATH,
        "evaluation",
    )

    log_existing_file(
        TIME_SERIES_FORECAST_PATH,
        "forecast",
    )

print("Final time-series experiment logged.")


Final time-series experiment logged.


# 4. Inspect the MLflow Experiment

The following cells read the runs back from MLflow.

Classification, regression, and time-series runs are inspected separately because their targets and metrics are different.


In [23]:
all_runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["start_time ASC"],
    output_format="pandas",
)

print("Total runs:", len(all_runs))

display(
    all_runs[
        [
            "run_id",
            "tags.task",
            "tags.stage",
            "tags.status",
            "tags.mlflow.runName",
            "metrics.cv_rmse",
            "metrics.tuned_rf_cv_rmse",
            "metrics.test_mae",
            "metrics.test_rmse",
            "metrics.test_r2",
            "metrics.validation_rmse",
        ]
    ]
)


Total runs: 48


,run_id,tags.task,tags.stage,tags.status,tags.mlflow.runName,metrics.cv_rmse,metrics.tuned_rf_cv_rmse,metrics.test_mae,metrics.test_rmse,metrics.test_r2,metrics.validation_rmse
0,a6ea0ee52c9d497caabf05117c9df3d5,classification,model_comparison,diagnostic,Classification - Random Forest CV,NaN,NaN,NaN,NaN,NaN,NaN
1,bd4c63d61a7e429c8f098fd27a2bcb6b,classification,model_comparison,diagnostic,Classification - Extra Trees CV,NaN,NaN,NaN,NaN,NaN,NaN
2,a349f349a5d7496abdcec98153446a36,classification,model_comparison,diagnostic,Classification - KNN CV,NaN,NaN,NaN,NaN,NaN,NaN
3,eaa55a0ad0884058a3a8c38624809540,classification,model_comparison,diagnostic,Classification - Gradient Boosting CV,NaN,NaN,NaN,NaN,NaN,NaN
4,2d26b66befe14e44ab588db3afaf1927,classification,model_comparison,diagnostic,Classification - XGBoost CV,NaN,NaN,NaN,NaN,NaN,NaN
5,6433b1b0d66f4e59a807bf758cfd74ec,classification,model_comparison,diagnostic,Classification - SVM CV,NaN,NaN,NaN,NaN,NaN,NaN
6,b9cbdbcee09d44d5943fd791a5493c76,classification,model_comparison,diagnostic,Classification - Logistic Regression CV,NaN,NaN,NaN,NaN,NaN,NaN
7,3320b165a9424ff6a308d32f93c8dae0,classification,model_comparison,diagnostic,Classification - Gaussian Naive Bayes CV,NaN,NaN,NaN,NaN,NaN,NaN
8,9a7ca01942044904aa7011ffcd9130fd,classification,model_comparison,diagnostic,Classification - Decision Tree CV,NaN,NaN,NaN,NaN,NaN,NaN
9,0ed8ff1f300e45d9941931d086dc2c3e,classification,final_test_evaluation,diagnostic,Classification - Random Forest Diagnostic Final,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# Regression run summary

regression_runs = all_runs[
    all_runs["tags.task"] == "regression"
].copy()

regression_columns = [
    "tags.mlflow.runName",
    "tags.stage",
    "tags.status",
    "metrics.cv_rmse",
    "metrics.cv_mae",
    "metrics.cv_r2",
    "metrics.tuned_rf_cv_rmse",
    "metrics.test_mae",
    "metrics.test_rmse",
    "metrics.test_r2",
]

available_columns = [
    column
    for column in regression_columns
    if column in regression_runs.columns
]

display(
    regression_runs[
        available_columns
    ].sort_values("tags.mlflow.runName")
)


,tags.mlflow.runName,tags.stage,tags.status,metrics.cv_rmse,metrics.cv_mae,metrics.cv_r2,metrics.tuned_rf_cv_rmse,metrics.test_mae,metrics.test_rmse,metrics.test_r2
12,Regression - Extra Trees CV,model_comparison,None,9.499911,8.136611,-0.045180,NaN,NaN,NaN,NaN
36,Regression - Extra Trees CV,model_comparison,None,9.499911,8.136611,-0.045180,NaN,NaN,NaN,NaN
13,Regression - Gradient Boosting CV,model_comparison,None,9.565338,8.188967,-0.059594,NaN,NaN,NaN,NaN
37,Regression - Gradient Boosting CV,model_comparison,None,9.565338,8.188967,-0.059594,NaN,NaN,NaN,NaN
15,Regression - Lasso CV,model_comparison,None,9.787639,8.308873,-0.109444,NaN,NaN,NaN,NaN
39,Regression - Lasso CV,model_comparison,None,9.787639,8.308873,-0.109444,NaN,NaN,NaN,NaN
17,Regression - Linear Regression CV,model_comparison,None,10.016905,8.453817,-0.162343,NaN,NaN,NaN,NaN
41,Regression - Linear Regression CV,model_comparison,None,10.016905,8.453817,-0.162343,NaN,NaN,NaN,NaN
10,Regression - Mean Baseline CV,model_comparison,None,9.314511,8.021754,-0.004751,NaN,NaN,NaN,NaN
34,Regression - Mean Baseline CV,model_comparison,None,9.314511,8.021754,-0.004751,NaN,NaN,NaN,NaN


In [25]:
# Time-series run summary

time_series_runs = all_runs[
    all_runs["tags.task"] == "time_series"
].copy()

time_series_columns = [
    "tags.mlflow.runName",
    "tags.stage",
    "tags.status",
    "metrics.validation_mae",
    "metrics.validation_rmse",
    "metrics.test_mae",
    "metrics.test_rmse",
]

available_columns = [
    column
    for column in time_series_columns
    if column in time_series_runs.columns
]

display(
    time_series_runs[
        available_columns
    ].sort_values("tags.mlflow.runName")
)


,tags.mlflow.runName,tags.stage,tags.status,metrics.validation_mae,metrics.validation_rmse,metrics.test_mae,metrics.test_rmse
23,Time Series - 3-Day Moving Average Final,final_test_evaluation,final,0.281121,0.401374,0.960359,1.094608
47,Time Series - 3-Day Moving Average Final,final_test_evaluation,final,0.281121,0.401374,0.960359,1.094608
19,Time Series - 3-Day Moving Average Validation,model_comparison,None,0.281121,0.401374,NaN,NaN
43,Time Series - 3-Day Moving Average Validation,model_comparison,None,0.281121,0.401374,NaN,NaN
21,"Time Series - ARIMA(1,0,0) Validation",model_comparison,None,0.486423,0.694097,NaN,NaN
45,"Time Series - ARIMA(1,0,0) Validation",model_comparison,None,0.486423,0.694097,NaN,NaN
22,Time Series - Naive Last Value Validation,model_comparison,None,0.583333,0.782152,NaN,NaN
46,Time Series - Naive Last Value Validation,model_comparison,None,0.583333,0.782152,NaN,NaN
20,Time Series - Simple Exponential Smoothing Val...,model_comparison,None,0.480235,0.682426,NaN,NaN
44,Time Series - Simple Exponential Smoothing Val...,model_comparison,None,0.480235,0.682426,NaN,NaN


In [26]:
# Save a CSV snapshot of the MLflow run history.

MLFLOW_SUMMARY_PATH = (
    MLFLOW_RESULTS_DIR
    / "mlflow_run_summary.csv"
)

all_runs.to_csv(
    MLFLOW_SUMMARY_PATH,
    index=False
)

print("MLflow run summary saved to:")
print(MLFLOW_SUMMARY_PATH)


MLflow run summary saved to:
C:\Users\srisa\OneDrive\Desktop\Projects\Public Transport Delays\results\mlflow\mlflow_run_summary.csv


# 5. Launch the MLflow UI

From the **project root** in a terminal, run:

```bash
mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000
```

Then open:

```text
http://127.0.0.1:5000
```

The UI lets you inspect:

- experiments
- runs
- parameters
- metrics
- model artefacts
- evaluation files
- explainability files

Each execution of this notebook creates another set of runs. This is expected behaviour for experiment tracking.


In [27]:
print("MLflow tracking database:", MLFLOW_DB_PATH)
print()
print("Launch from the project root:")
print("mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000")
print()
print("Open:")
print("http://127.0.0.1:5000")


MLflow tracking database: C:\Users\srisa\OneDrive\Desktop\Projects\Public Transport Delays\mlflow.db

Launch from the project root:
mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000

Open:
http://127.0.0.1:5000


# 6. Automated Verification

The checks below confirm that the MLflow experiment contains the expected task types and final runs, and that the source result files are available.

The regression model and CSV consistency checks were already performed before logging the final regression run.


In [28]:
verification = {
    "classification_runs_present": (
        (all_runs["tags.task"] == "classification").any()
    ),
    "regression_runs_present": (
        (all_runs["tags.task"] == "regression").any()
    ),
    "time_series_runs_present": (
        (all_runs["tags.task"] == "time_series").any()
    ),
    "final_regression_run_present": (
        (
            (all_runs["tags.task"] == "regression")
            & (all_runs["tags.status"] == "final")
        ).any()
    ),
    "final_time_series_run_present": (
        (
            (all_runs["tags.task"] == "time_series")
            & (all_runs["tags.status"] == "final")
        ).any()
    ),
    "classification_cv_file_exists": (
        CLASSIFICATION_CV_PATH.exists()
    ),
    "classification_test_file_exists": (
        CLASSIFICATION_TEST_PATH.exists()
    ),
    "regression_cv_file_exists": (
        REGRESSION_CV_PATH.exists()
    ),
    "regression_tuning_file_exists": (
        REGRESSION_TUNING_PATH.exists()
    ),
    "regression_test_file_exists": (
        REGRESSION_TEST_PATH.exists()
    ),
    "time_series_validation_file_exists": (
        TIME_SERIES_VALIDATION_PATH.exists()
    ),
    "time_series_test_file_exists": (
        TIME_SERIES_TEST_PATH.exists()
    ),
    "mlflow_summary_exists": (
        MLFLOW_SUMMARY_PATH.exists()
    ),
    "tracking_database_exists": (
        MLFLOW_DB_PATH.exists()
    ),
}

for name, passed in verification.items():
    print(f"{name}: {passed}")

if not all(verification.values()):
    raise AssertionError(
        "One or more MLflow verification checks failed."
    )

print()
print("All MLflow verification checks passed.")


classification_runs_present: True
regression_runs_present: True
time_series_runs_present: True
final_regression_run_present: True
final_time_series_run_present: True
classification_cv_file_exists: True
classification_test_file_exists: True
regression_cv_file_exists: True
regression_tuning_file_exists: True
regression_test_file_exists: True
time_series_validation_file_exists: True
time_series_test_file_exists: True
mlflow_summary_exists: True
tracking_database_exists: True

All MLflow verification checks passed.


# MLflow Experiment Tracking — Final Conclusion

MLflow was integrated as an experiment-tracking layer over the completed classification, regression, and time-series workflows.

The modelling notebooks remain the source of truth. Their generated result files are read directly by this notebook, preventing manual re-entry of metrics and hyperparameters.

The classification workflow is recorded as a diagnostic experiment because of the previously identified target-construction limitation. The regression workflow records candidate-model cross-validation results, Random Forest tuning parameters, final test metrics, the saved model, and explainability artefacts. The time-series workflow records chronological validation experiments and the selected model's final test evaluation.

Before logging the final regression run, the saved Random Forest pipeline was used to recompute the test metrics and verify that they matched the metrics stored by Notebook 4. The saved tuning parameters were also checked against the fitted Random Forest.

This creates a reproducible experiment history containing **parameters, metrics, models, evaluation files, and explainability artefacts**, providing the foundation for the next MLOps stage: reusable prediction code, serving, deployment, Docker, and CI/CD.
